# Demo 2: Speculative Decoding 模拟

模拟 Speculative Decoding 的加速效果，对比不同 K 值和接受率下的加速比。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import numpy as np

# 注册中文字体
fm.fontManager.addfont('/home/zhangboju/.fonts/NotoSansSC.ttf')

matplotlib.rcParams.update({
    'figure.facecolor': '#1a2332',
    'axes.facecolor': '#1a2332',
    'axes.edgecolor': '#8faab0',
    'axes.labelcolor': '#f1faee',
    'text.color': '#f1faee',
    'xtick.color': '#8faab0',
    'ytick.color': '#8faab0',
    'grid.color': '#223044',
    'legend.facecolor': '#223044',
    'legend.edgecolor': '#8faab0',
    'font.size': 12,
    'font.family': 'Noto Sans SC',
    'axes.unicode_minus': False,
})

def expected_speedup(alpha, K, c=0.1):
    """
    计算期望加速比
    alpha: 接受率
    K: draft 长度
    c: 小模型/大模型速度比 (小模型快多少倍)
    """
    # 期望接受的 token 数
    if alpha >= 1.0:
        return K
    expected_accepted = alpha * (1 - alpha**K) / (1 - alpha)
    # 每步的实际代价: K*c (draft) + 1 (verify)
    cost_per_step = K * c + 1
    # 无投机解码时每 token 成本为 1
    return expected_accepted / cost_per_step

# 不同参数下的加速比
K_values = [1, 2, 3, 4, 5, 8, 10]
alphas = [0.6, 0.7, 0.8, 0.9, 0.95]

fig, ax = plt.subplots(figsize=(9, 5.5))

colors = ['#e09060', '#e0b060', '#2d8b8b', '#5ec4c4', '#a8dadc']
for alpha, color in zip(alphas, colors):
    speedups = [expected_speedup(alpha, K) for K in K_values]
    ax.plot(K_values, speedups, 'o-', color=color, linewidth=2, markersize=6,
            label=f'α = {alpha}')

ax.set_xlabel('Draft Length K')
ax.set_ylabel('Expected Speedup')
ax.set_title('Speculative Decoding Speedup', fontsize=14, color='#a8dadc', pad=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='#8faab0', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../assets/chart-spec-decode.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart-spec-decode.png')